# Accessibilité — Abidjan (AMUGA), grille WorldPop

Équivalent de [`index_accessibility_notebook_def.ipynb`](https://github.com/antoinechevre/Accessibility_analysis/blob/main/index_accessibility_notebook_def.ipynb) du projet sœur [Accessibility_analysis](https://github.com/antoinechevre/Accessibility_analysis), mais pour un réseau hors de France (ici Abidjan / AMUGA) :

- **Population** : grille WorldPop sur le rectangle englobant les arrêts du GTFS (`construire_grille_population_gtfs`, cf. `world_pop_data.ipynb`) au lieu du carroyage 200x200 INSEE (France uniquement).
- **Réseau de routage** : extrait OSM téléchargé via Overpass sur cette même zone (`src/osm_extract.py`, générique — pas de dépendance à un découpage communal français) au lieu de l'extraction basée sur les codes commune INSEE.
- **Équipements (BPE)** : la Base Permanente des Équipements est un référentiel INSEE, propre à la France — pas d'équivalent branché pour la Côte d'Ivoire. Substitut : équipements OpenStreetMap pondérés par type (`extraire_amenities_osm.py` + référentiel `data/equipements_osm/Abidjan_amenities.xlsx`), dispatchés sur la grille en un score unique `land_use_data["equipements"]`, **sans découpage par domaine** (une seule source, contrairement aux domaines BPE A-G du notebook source). `cumulative_cutoff` (3.2.1), `cost_to_closest` (3.2.2), `gravity` (3.2.3), Enhanced 2SFCA (3.2.4) et la comparaison de seuils (3.2.5) sont toutes portées sur ce score. Restent **vides**, faute d'équivalent hors de France : tableaux de pôles d'équipements par domaine, benchmark par domaine, inégalités par décile de niveau de vie Filosofi (INSEE).
- Le reste (temps de trajet TC+marche via [r5py](https://r5py.readthedocs.io/)) est fonctionnel.

In [6]:
#réinitialise module

%load_ext autoreload
%autoreload 2

import os

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"

# Fix : forcer l'initialisation de PROJ/GDAL avec les données de rasterio AVANT
# import r5py (même raison que index_accessibility_notebook_def.ipynb : le
# démarrage de la JVM par r5py écrase sinon PROJ_LIB avec un chemin invalide).
import rasterio

import r5py
import r5py.util.jvm
r5py.util.jvm.MAX_JVM_MEMORY = 2 * 1024**3  # 2 Go, à remonter si vous avez de la RAM libre

import shutil
import time
import datetime

import pandas as pd
import geopandas as gpd
import folium

import src.info_reseau as _info_reseau
from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    dir_tree,
    preparer_gtfs_pour_r5py,
)
from src.hf_cache import envoyer_vers_hf, recuperer_depuis_hf
from src.worldpop import construire_grille_population_gtfs, carte_population_worldpop, zone_desservie_gtfs
from src.osm_extract import osm_pbf_creator_depuis_geometrie
from src.utilitaires_matrix import calculer_ttm_par_lots, charger_ttm, cumulative_cutoff

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
#chemins fixes

BASE_DIR = os.getcwd()

GTFS_ZIP_PATH = os.path.join(BASE_DIR, "data", "GTFS_Africa", "Abidjan_AMUGA_GTFS_2025_mapping_v2.zip")  # <- à adapter

MEMORY_TTM_DIR = os.path.join(BASE_DIR, "data", "memory_ttm")
MEMORY_PBF_DIR = os.path.join(BASE_DIR, "data", "memory_pbf")
DOSSIER_CACHE_GTFS = os.path.join(BASE_DIR, "data", "worldpop")  # rasters pays WorldPop, gitignoré

output_path = os.path.join(BASE_DIR, "output")
data_path = os.path.join(BASE_DIR, "data")

FONDS_CARTE = {
    "OpenStreetMap": "OpenStreetMap",
    "CartoDB Positron": "CartoDB positron",
    "CartoDB Dark Matter": "CartoDB dark_matter",
}
FOND_CARTE = "CartoDB Positron"

print(BASE_DIR)
print(GTFS_ZIP_PATH)

/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app
/Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/GTFS_Africa/Abidjan_AMUGA_GTFS_2025_mapping_v2.zip


## Charger le GTFS

In [8]:
#charge GTFS en feed et nom du réseau

feed = charger_gtfs(GTFS_ZIP_PATH)
print(type(feed))

nb_agences = len(feed.agency)
if nb_agences > 3:
    print(f"⚠ Ce GTFS regroupe {nb_agences} agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement.")

# On appelle via le module (pas le nom importé) pour ne jamais l'écraser avec
# son résultat : sinon, réexécuter cette cellule une deuxième fois lève
# "TypeError: 'str' object is not callable".
nom_reseau_str = _info_reseau.nom_reseau_str(feed)
print(nom_reseau_str)

dates_service_list, date_debut, date_fin, date_JOB = _info_reseau.dates_service(feed)
print(dates_service_list, date_debut, date_fin, date_JOB)

# Véhicules.km pour la date JOB (générique, indépendant de la BPE)
longueur_par_ligne = longueur_lignes(feed)
vkm_par_ligne_job = km_par_ligne_jour(feed, longueur_par_ligne, date_JOB)
total_vkm_job = vkm_par_ligne_job["total_km"].sum()
print(f"Véhicules.km le {date_JOB} (JOB) : {total_vkm_job:,.0f} km, sur {len(vkm_par_ligne_job)} ligne(s)")
vkm_par_ligne_job.sort_values("total_km", ascending=False)

Chargement du fichier GTFS : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/GTFS_Africa/Abidjan_AMUGA_GTFS_2025_mapping_v2.zip
✓ GTFS chargé avec succès
<class 'gtfs_kit.feed.Feed'>
⚠ Ce GTFS regroupe 8 agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement.
AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba…
['20250501', '20250502', '20250503', '20250504', '20250505', '20250506', '20250507', '20250508', '20250509', '20250510', '20250511', '20250512', '20250513', '20250514', '20250515', '20250516', '20250517', '20250518', '20250519', '20250520', '20250521', '20250522', '20250523', '20250524', '20250525', '20250526', '20250527', '20250528', '20250529', '20250530', '20250531', '20250601', '20250602', '20250603', '20250604', '20250605', '20250606', '20250607', '20250608', '20250609', '20250610', '20250611', '20250612', '20250613', '20250614', '20250615', '20250616', '20250617', '20250618', '20250619', 

,route_id,total_km,date
179,Gare de bassam vers gare bonoua,625.299052,20281228
633,rond point samake abobo vers Nouvelle gare de ...,533.905890,20281228
164,Gare bonoua vers village hono,515.129441,20281228
183,Gare jacqueville apache vers treichville avenu...,508.888983,20281228
363,adjamé mosquée vers gare jacqueville,493.222308,20281228
...,...,...,...
386,attecoube vers mosquée adjame,7.004424,20281228
324,abobo anokoua-pk18 abobo,5.388920,20281228
377,angre petro ivoire cocody vers terminus 81_82 ...,3.976486,20281228
344,académie yopougon vers fin goudron yopougon,3.647674,20281228


## Construction de la grille de population (WorldPop)

Même zone que `world_pop_data.ipynb` : rectangle englobant tous les arrêts du GTFS, avec une marge de sécurité `MARGE_KM`. `population_grid_agglo`/`land_use_data` (mêmes noms que le notebook source) servent de grille de référence pour toute la suite — origines/destinations du routage, "opportunité" population pour `cumulative_cutoff`.

In [9]:
MARGE_KM = 5  # marge de sécurité ajoutée sur chaque côté du rectangle englobant les arrêts, en km
ANNEE_GTFS = 2020  # dernière année disponible dans le dataset WorldPop "Unconstrained individual countries" (2000-2020)
RESOLUTION_M_GTFS = 400  # taille de carreau cible en mètres

population_grid_agglo, lat_centre_gtfs, lon_centre_gtfs = construire_grille_population_gtfs(
    feed,
    marge_km=MARGE_KM,
    annee=ANNEE_GTFS,
    resolution_m=RESOLUTION_M_GTFS,
    dossier_cache=DOSSIER_CACHE_GTFS,
)

NOM_ZONE_GTFS = os.path.splitext(os.path.basename(GTFS_ZIP_PATH))[0]

# land_use_data : GeoDataFrame de travail pour cumulative_cutoff etc. (mêmes
# noms de colonnes que le notebook source : id, population)
land_use_data = population_grid_agglo[["id", "population"]].copy()

print(f"{len(population_grid_agglo)} carreaux, population totale : {population_grid_agglo['population'].sum():,.0f} habitants")
population_grid_agglo.head()

✓ Zone GTFS : 1116 arrêts, rectangle centré sur 5.4048, -4.0618
✓ Pays couverts par la zone GTFS : ['CIV']
✓ Raster déjà en cache : /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/worldpop/CIV_ppp_2020.tif
✓ CIV : 59273 carreaux dans la zone
59273 carreaux, population totale : 6,153,925 habitants


,population,geometry,id
0,3.490534,"POLYGON ((-4.76542 5.68625, -4.76542 5.68292, ...",0
1,3.424140,"POLYGON ((-4.76208 5.68625, -4.76208 5.68292, ...",1
2,3.283505,"POLYGON ((-4.75875 5.68625, -4.75875 5.68292, ...",2
3,3.266769,"POLYGON ((-4.75542 5.68625, -4.75542 5.68292, ...",3
4,3.380386,"POLYGON ((-4.75208 5.68625, -4.75208 5.68292, ...",4


## Extraction OSM pour le réseau de routage

Même rectangle (arrêts GTFS + marge) que la grille de population ci-dessus, pour que le réseau routier couvre au moins toute la zone analysée. Téléchargement via Overpass, tuile par tuile, puis découpage précis avec osmium (`src/osm_extract.py`, cf. `osm_pbf_creator` du notebook source — déjà générique, adapté ici pour partir directement d'une géométrie plutôt que d'un GeoJSON de communes).

⚠ Premier appel : peut prendre plusieurs minutes (Overpass + osmium). Mis en cache localement (`data/memory_pbf/`) et sur Hugging Face (`memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf`, namespacé par réseau) : les appels suivants sont immédiats.

In [10]:
zone_geom, _, _ = zone_desservie_gtfs(feed, marge_km=MARGE_KM)

os.makedirs(MEMORY_PBF_DIR, exist_ok=True)
PBF_PATH_SAVED = os.path.join(MEMORY_PBF_DIR, f"agglo_osm_pbf_{nom_reseau_str}.osm.pbf")
recuperer_depuis_hf(f"memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf", PBF_PATH_SAVED)

OSM_WORK_DIR = os.path.join(data_path, "osm_extract")
AGGLO_PBF_PATH = os.path.join(OSM_WORK_DIR, "agglo.osm.pbf")

if os.path.exists(PBF_PATH_SAVED):
    os.makedirs(OSM_WORK_DIR, exist_ok=True)
    shutil.copyfile(PBF_PATH_SAVED, AGGLO_PBF_PATH)
    print(f"extrait OSM déjà présent pour ce réseau, copié depuis {PBF_PATH_SAVED}")
else:
    AGGLO_PBF_PATH = osm_pbf_creator_depuis_geometrie(zone_geom, OSM_WORK_DIR)
    shutil.copyfile(AGGLO_PBF_PATH, PBF_PATH_SAVED)
    envoyer_vers_hf(PBF_PATH_SAVED, f"memory_pbf/agglo_osm_pbf_{nom_reseau_str}.osm.pbf")

print(AGGLO_PBF_PATH)

[hf_cache] recuperer_depuis_hf('memory_pbf/agglo_osm_pbf_AMUGA - Gbaka 10 à 14 places - Gbaka 18 à 22 places - Gbaka 26 à 32 places - Gba….osm.pbf') absent de antoinechevre/ww_GTFS : RemoteEntryNotFoundError('404 Client Error. (Request ID: Root=1-6a8c427f-2f41283c0164454343aa317a;9e6aa078-3aee-480a-9ec4-9aae5f5fa567)\n\nEntry Not Found for url: https://huggingface.co/datasets/antoinechevre/ww_GTFS/resolve/main/memory_pbf/agglo_osm_pbf_AMUGA%20-%20Gbaka%2010%20%C3%A0%2014%20places%20-%20Gbaka%2018%20%C3%A0%2022%20places%20-%20Gbaka%2026%20%C3%A0%2032%20places%20-%20Gba%E2%80%A6.osm.pbf.')
wrote /Users/antoinechevre/Documents/_0_Programme/6_GTFS_universal/gtfs_analysis_app/data/osm_extract/agglo_boundary.geojson, bounds: [-4.76502429  5.12394926 -3.35856977  5.68561713]
emprise découpée en 10 tuile(s) de 0.3° pour Overpass (essai 1/3)
téléchargement tuile 1/10 (bbox (np.float64(-4.765024286964603), np.float64(5.1239492636297115), np.float64(-4.465024286964603), np.float64(5.4239492636297

KeyboardInterrupt: 

## Équipements (substitut OSM pondéré à la BPE)

Pas de Base Permanente des Équipements pour la Côte d'Ivoire : extraction de tous les `amenity=*` OpenStreetMap (`extraire_amenities_osm.py`) sur la même zone que la grille de population (`zone_geom`), pondérés par type via le référentiel défini à la main sur Abidjan (`data/equipements_osm/Abidjan_amenities.xlsx`).

In [ ]:
from extraire_amenities_osm import extraire_amenities_osm

# Substitut à la BPE (pas de données INSEE pour la Côte d'Ivoire) : tous les
# équipements OpenStreetMap taggés amenity=* sur la même zone que la grille
# de population (zone_geom), pondérés par type d'équipement — pondération
# définie à la main sur Abidjan (data/equipements_osm/Abidjan_amenities.xlsx,
# feuille "resume_par_type", colonne "Ponderation" : 0 = pas un pôle
# d'équipement pertinent, jusqu'à 30 pour les équipements structurants comme
# hôpital/gouvernement/bâtiment public) et réutilisée telle quelle comme
# référentiel unique pour toutes les villes — un type d'amenity absent de ce
# référentiel (non rencontré sur Abidjan) reçoit une pondération de 0, pas
# d'erreur. Remplace l'isopondération précédente (chaque équipement comptait
# pour 1, quel que soit son type).

DOSSIER_EQUIPEMENTS_OSM = os.path.join(data_path, "equipements_osm")
PONDERATION_XLSX = os.path.join(DOSSIER_EQUIPEMENTS_OSM, "Abidjan_amenities.xlsx")  # référentiel partagé, cf. ci-dessus

amenities = extraire_amenities_osm(NOM_ZONE_GTFS, zone_geom=zone_geom)

ponderation_par_amenity = pd.read_excel(PONDERATION_XLSX, sheet_name="resume_par_type").set_index("amenity")["Ponderation"]
amenities["ponderation"] = amenities["amenity"].map(ponderation_par_amenity).fillna(0)

nb_hors_referentiel = amenities.loc[~amenities["amenity"].isin(ponderation_par_amenity.index), "amenity"].nunique()
print(f"✓ {len(amenities)} amenity(s) extrait(s), {(amenities['ponderation'] > 0).sum()} avec une pondération > 0")
if nb_hors_referentiel:
    print(f"  ({nb_hors_referentiel} type(s) d'amenity absent(s) du référentiel Abidjan, pondérés à 0 par défaut)")

# Sauvegarde en un seul fichier consolidé (un point = un équipement, avec sa
# pondération), réutilisable tel quel par l'app (views/equipements.py,
# views/accessibilite.py) — remplace les anciens fichiers par catégorie
# isopondérés (abidjan_hopitaux.gpkg etc.), désormais désynchronisés de
# cette logique pondérée.
os.makedirs(DOSSIER_EQUIPEMENTS_OSM, exist_ok=True)
NOM_VILLE_SIMPLE = NOM_ZONE_GTFS.split("_")[0]
CHEMIN_EQUIPEMENTS_GPKG = os.path.join(DOSSIER_EQUIPEMENTS_OSM, f"{NOM_VILLE_SIMPLE.lower()}_equipements.gpkg")

amenities_geo = gpd.GeoDataFrame(
    amenities, geometry=gpd.points_from_xy(amenities["lon"], amenities["lat"]), crs="EPSG:4326",
)
amenities_geo.to_file(CHEMIN_EQUIPEMENTS_GPKG, driver="GPKG")
print(f"✓ Équipements sauvegardés : {CHEMIN_EQUIPEMENTS_GPKG}")

amenities.loc[amenities["ponderation"] > 0].groupby("amenity")["ponderation"].first().sort_values(ascending=False)

## Dispatch pondéré des équipements sur la grille (scoring BPE)

Jointure spatiale des équipements extraits ci-dessus dans les carreaux 400x400 de `population_grid_agglo`, pondérés par type (`land_use_data["equipements"]` = somme des pondérations, pas un simple comptage).

In [ ]:
# Dispatch pondéré des équipements sur les carreaux 400x400 de
# population_grid_agglo (jointure spatiale point-dans-polygone) :
# land_use_data["equipements"] est désormais une somme pondérée par carreau
# (le "scoring BPE" de ce notebook), pas un simple comptage isopondéré.
jointure = gpd.sjoin(amenities_geo, population_grid_agglo[["id", "geometry"]], how="inner", predicate="within")
score_par_carreau = jointure.groupby("id")["ponderation"].sum().rename("equipements")

land_use_data = land_use_data.merge(score_par_carreau, on="id", how="left")
land_use_data["equipements"] = land_use_data["equipements"].fillna(0)

print(f"✓ score total : {land_use_data['equipements'].sum():,.0f} (pondéré), sur {len(land_use_data)} carreaux")
land_use_data.sort_values("equipements", ascending=False).head()

## Construction du réseau de transport multimodal (r5py)

Équivalent de `setup_r5(data_path)` : l'objet `TransportNetwork` joue à la fois le rôle du réseau construit et du point d'entrée pour les calculs de temps de trajet (`TravelTimeMatrix`).

In [ ]:
print(data_path)
dir_tree(data_path)

GTFS_PATH_R5PY = preparer_gtfs_pour_r5py(GTFS_ZIP_PATH)

try:
    transport_network = r5py.TransportNetwork(
        osm_pbf=str(AGGLO_PBF_PATH),
        gtfs=[str(GTFS_PATH_R5PY)],
    )
except Exception:
    # r5py met en cache dans Config().CACHE_DIR (~/.cache/r5py) le graphe déjà
    # construit ET une copie de travail de l'OSM pbf : un run précédent
    # interrompu peut y laisser un fichier à moitié écrit, qui ne se répare
    # jamais tout seul (cf. index_accessibility_notebook_def.ipynb, même
    # correctif). On vide ce cache (SAUF les .jar R5, ~65 Mo, déjà
    # téléchargés une fois pour toutes) et on relance une fois.
    from r5py.util import Config

    cache_dir = Config().CACHE_DIR
    for entree in cache_dir.iterdir():
        if entree.suffix == ".jar":
            continue
        if entree.is_dir():
            shutil.rmtree(entree, ignore_errors=True)
        else:
            entree.unlink(missing_ok=True)
    transport_network = r5py.TransportNetwork(
        osm_pbf=str(AGGLO_PBF_PATH),
        gtfs=[str(GTFS_PATH_R5PY)],
    )

population_grid_agglo.info()

## Matrice de temps de trajet (TTM)

Origines/destinations = centroïdes de la grille de population. Calculée par lots (`calculer_ttm_par_lots`) pour borner le pic mémoire, écrite sur disque au fur et à mesure. Mise en cache localement et sur Hugging Face (`memory_ttm/ttm_{nom_reseau_str}.parquet`), rechargée directement si calculée il y a moins de 10 jours.

In [ ]:
points = population_grid_agglo[["id", "geometry"]].copy()
points["geometry"] = points.geometry.centroid

os.makedirs(MEMORY_TTM_DIR, exist_ok=True)
TTM_PATH = os.path.join(MEMORY_TTM_DIR, f"ttm_{nom_reseau_str}.parquet")

recuperer_depuis_hf(f"memory_ttm/ttm_{nom_reseau_str}.parquet", TTM_PATH)

ttm_cache_recent = (
    os.path.exists(TTM_PATH) and (time.time() - os.path.getmtime(TTM_PATH)) < 10 * 24 * 3600
)

if ttm_cache_recent:
    ttm = charger_ttm(TTM_PATH)
    print(f"ttm rechargé depuis le cache (< 10 jours) : {TTM_PATH}")
else:
    departure_datetime = datetime.datetime.strptime(date_JOB, "%Y%m%d").replace(
        hour=14, minute=0, second=0
    )

    calculer_ttm_par_lots(
        r5py,
        transport_network,
        points,
        departure=departure_datetime,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        max_time_walking=datetime.timedelta(minutes=30),
        max_time=datetime.timedelta(minutes=120),
        ttm_path=TTM_PATH,
        on_step=print,
    )
    ttm = charger_ttm(TTM_PATH)

    envoyer_vers_hf(TTM_PATH, f"memory_ttm/ttm_{nom_reseau_str}.parquet")

ttm.head()

## 3.2.1 Mesure des opportunités cumulées (`cumulative_cutoff`)

Seule mesure du notebook source qui ne dépend pas d'une source d'équipements : `opportunity="population"` mesure le nombre d'habitants accessibles en <= `cutoff` minutes depuis chaque carreau. 3.2.2 à 3.2.5 sont portées plus bas avec `opportunity="equipements"` (score OSM pondéré, cf. section "Dispatch pondéré des équipements sur la grille" plus haut) — sans découpage par domaine (une seule source d'équipements pour la Côte d'Ivoire, contrairement à la BPE du notebook source).

In [ ]:
cum_opportunities = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="population",
    travel_cost="travel_time",
    cutoff=30,
)

cum_opportunities.head()

## 3.2.2 Coût de trajet minimum (`cost_to_closest`)

Contrairement au notebook source (domaine BPE `DOMAINE_CIBLE`, `land_use_data_domaine`), il n'y a ici qu'un seul score d'équipements, tous types OSM confondus (`land_use_data["equipements"]`, cellule "Dispatch pondéré..." ci-dessus) — pas de découpage par domaine pour la Côte d'Ivoire. On passe donc `land_use_data` directement à `cost_to_closest` (`opportunity="equipements"`), ce qui court-circuite la construction automatique par domaine BPE : les 4 premiers paramètres (`land_use_data_domaine`, `BPE_agglo`, `_land_use_data_global`, `DOMAINES_BPE`) ne sont alors jamais utilisés, on peut leur passer `None`.

In [ ]:
from src.utilitaires_matrix import cost_to_closest

min_time_equipements = cost_to_closest(
    None, None, None, {},  # land_use_data_domaine, BPE_agglo, _land_use_data_global, DOMAINES_BPE : jamais utilisés puisque land_use_data est fourni directement ci-dessous (DOMAINES_BPE reste un dict vide, pas None : cost_to_closest y fait DOMAINES_BPE.get(...) même dans ce cas, pour le message affiché)
    ttm,
    opportunity="equipements",
    travel_cost="travel_time",
    land_use_data=land_use_data,
    n=1,  # atteint dès qu'un carreau de score pondéré >= 1 est accessible (pas de notion de "pôle" par domaine ici)
)

min_time_equipements.head()

## 3.2.3 Mesures de gravité (`gravity`)

Comme pour 3.2.2 : pas de domaine BPE, `opportunity="equipements"` directement sur `land_use_data` (score OSM pondéré, tous types confondus) — `gravity()` n'a de toute façon aucune dépendance à la BPE dans sa signature, contrairement à `cost_to_closest`.

In [ ]:
from src.utilitaires_matrix import decay_exponential, gravity

negative_exp_grav = gravity(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    decay_function=decay_exponential(0.2),
)

negative_exp_grav.head()

## 3.2.4 Mesures de compétition (Enhanced 2SFCA)

Pas d'implémentation fidèle du BFCA (Paez, Higgins & Vivona 2019, algorithme d'équilibrage itératif offre/demande, trop spécifique pour être reconstitué de mémoire) — Enhanced 2SFCA (Luo & Qi, 2009) à la place, comme dans le notebook source : ratio offre/demande pondéré par la décroissance, en deux étapes, sans boucle d'équilibrage. Les valeurs ne correspondent pas exactement à `{accessibility}::floating_catchment_area(method = "bfca")`.

`supply` = `equipements`, `demand` = `population`, tous deux déjà dans `land_use_data` (pas de `land_use_data_domaine` ni de merge à faire, contrairement au notebook source). `enhanced_2sfca_par_lots()` plutôt que `enhanced_2sfca()` : relit `ttm` depuis `TTM_PATH` par row group (pyarrow) plutôt que de dupliquer la matrice en mémoire via deux `.merge()` — même précaution mémoire que le notebook source (utile là-bas sur Lyon/TCL, moins critique ici sur la grille WorldPop 400m d'Abidjan, mais résultat identique et sans coût supplémentaire).

In [ ]:
from src.utilitaires_matrix import enhanced_2sfca_par_lots

e2sfca_equipements = enhanced_2sfca_par_lots(
    TTM_PATH,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    demand="population",
    decay_function=decay_exponential(0.05),
    on_step=print,
)

e2sfca_equipements.head()

## 3.2.5 Comparaison des seuils <= vs < (`cutoff`)

Même illustration que le notebook source (`cumulative_cutoff()` compte les trajets `<= cutoff`, pas `< cutoff` — `cutoff=29` équivaut donc à un seuil "moins de 30 minutes"), appliquée à `opportunity="equipements"` directement sur `land_use_data` (pas de domaine BPE à choisir, cf. 3.2.2 ci-dessus).

In [ ]:
cum_cutoff_30 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    cutoff=30,
)

cum_cutoff_29 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="equipements",
    travel_cost="travel_time",
    cutoff=29,
)

cutoff_comparison = cum_cutoff_30.merge(
    cum_cutoff_29, on="id", suffixes=("_cutoff_30_inclus", "_cutoff_29_soit_moins_de_30min")
)

cutoff_comparison.head()

## Carte de la grille de population

In [ ]:
os.makedirs(os.path.join(output_path, nom_reseau_str), exist_ok=True)
output_html_worldpop = os.path.join(output_path, nom_reseau_str, f"{NOM_ZONE_GTFS}_worldpop_gtfs_{MARGE_KM}km.html")

carte_pop = carte_population_worldpop(population_grid_agglo, NOM_ZONE_GTFS, ANNEE_GTFS, tiles=FONDS_CARTE[FOND_CARTE])
carte_pop.save(output_html_worldpop)
print(f"✓ Carte enregistrée : {output_html_worldpop}")

carte_pop

## Analyse accessibilité / pop par domaine + inégalités par niveau de vie (à faire plus tard)

Correspond à la section "analyse accessibilite / pop" du notebook source (9.1 temps d'accès au pôle le plus proche par domaine, 9.2 pôles accessibles à 30/45 min, 9.3 inégalités par décile de niveau de vie). Doublement bloqué pour la Côte d'Ivoire : boucle sur les domaines BPE (cf. ci-dessus) ET décile de niveau de vie calculé sur `ind_snv` (Filosofi, INSEE — France uniquement, pas de colonne équivalente dans la grille WorldPop).

## Tableaux récapitulatifs pôles d'équipements (à faire plus tard)

Correspond aux tableaux `pct_poles_atteignables_par_carreau`/`moyenne_ponderee_pct_poles` du notebook source (% de pôles d'équipements majeurs atteignables par domaine et par durée) — BPE requise.

## Sauvegarde index benchmark (à faire plus tard)

Correspond à `calculer_index_benchmark` du notebook source (indicateurs de synthèse par domaine BPE et par décile de niveau de vie, agrégés dans le CSV de benchmark inter-réseaux) — BPE et décile de niveau de vie requis, cf. sections précédentes.